In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import pypulseq as pp
import MRzeroCore as mr0
import torchkbnufft as tkbn
from pypulseq import Sequence
from torch 
torch.set_default_device("cuda")

In [ ]:
def main(
    plot: bool, 
    write_seq: bool,  
    fov = 260e-3
    Nx = 160  
    alpha = 5  
    slice_thickness = 3e-3  
    TE = 4e-3  
    TR = 100e-3  
    Nr = 256  
    N_dummy = 20  
    delta = np.pi / Nr   
): -> pp.Sequence:

    system = default_system()
    seq = pp.Sequence(system)
    rf_spoiling_inc = 117 

    rf, gz, _ = pp.make_sinc_pulse(
        apodization=0.5,
        duration=2e-3,
        flip_angle=alpha * np.pi / 180,
        slice_thickness=slice_thickness,
        system=system,
        time_bw_product=4,
        return_gz=True,
    )

    deltak = 1 / fov
    gx = pp.make_trapezoid(
        channel="x", flat_area=Nx * deltak, flat_time=6.4e-3 / 5, system=system
    )
    adc = pp.make_adc(
        num_samples=Nx, duration=gx.flat_time, delay=gx.rise_time, system=system
    )
    gx_pre = pp.make_trapezoid(
        channel="x", area=-gx.area / 2, duration=2e-3, system=system
    )
    gz_reph = pp.make_trapezoid(
        channel="z", area=-gz.area / 2, duration=2e-3, system=system
    )

    gx_spoil = pp.make_trapezoid(channel="x", area=0.5 * Nx * deltak, system=system)
    gz_spoil = pp.make_trapezoid(channel="z", area=4 / slice_thickness, system=system)

    delay_TE = (
        np.ceil(
            (
                TE
                - pp.calc_duration(gx_pre)
                - gz.fall_time
                - gz.flat_time / 2
                - pp.calc_duration(gx) / 2
            )
            / seq.grad_raster_time
        )
        * seq.grad_raster_time
    )
    delay_TR = (
        np.ceil(
            (
                TR
                - pp.calc_duration(gx_pre)
                - pp.calc_duration(gz)
                - pp.calc_duration(gx)
                - delay_TE
            )
            / seq.grad_raster_time
        )
        * seq.grad_raster_time
    )
    assert np.all(delay_TR) > pp.calc_duration(gx_spoil, gz_spoil)
    rf_phase = 0
    rf_inc = 0

    for i in range(-N_dummy, Nr + 1):
        rf.phase_offset = rf_phase / 180 * np.pi
        adc.phase_offset = rf_phase / 180 * np.pi

        rf_inc = divmod(rf_inc + rf_spoiling_inc, 360.0)[1]
        rf_phase = divmod(rf_inc + rf_phase, 360.0)[1]

        seq.add_block(rf, gz)
        phi = delta * (i - 1)
        seq.add_block(*pp.rotate(gx_pre, gz_reph, angle=phi, axis="z"))
        seq.add_block(pp.make_delay(delay_TE))
        if i > 0:
            seq.add_block(*pp.rotate(gx, adc, angle=phi, axis="z"))
        else:
            seq.add_block(*pp.rotate(gx, angle=phi, axis="z"))
        seq.add_block(
            *pp.rotate(gx_spoil, gz_spoil, pp.make_delay(delay_TR), angle=phi, axis="z")
        )

    ok, error_report = seq.check_timing()
    if ok:
        print("Timing check passed successfully")
    else:
        print("Timing check failed! Error listing follows:")
        print(error_report)

    if plot:
        seq.plot()

    if write_seq:
        seq.set_definition("FOV", [fov, fov, slice_thickness])
        seq.set_definition("Name", "gre_rad")
        seq.write("Radial 2D GRE")


if __name__ == "__main__":
    main(plot=True, write_seq=True)

In [ ]:
mr_seq = mr0.Sequence.import_file("Radial 2D GRE.seq")
signal,_ = mr0.util.simulate("Radial 2D GRE.seq", None)
ktraj_raw = mr_seq.get_kspace()[:, :2].T
ktraj_torch = torch.tensor(ktraj_raw).float()
k_max = ktraj_torch.abs().max()
ktraj_torch = (ktraj_torch / k_max) * np.pi
ktraj_torch = ktraj_torch.unsqueeze(0)
adjnufft_ob = tkbn.KbNufftAdjoint(im_size=(320, 320), grid_size=(640, 640)).to(ktraj_torch.device)
rho = torch.sqrt(ktraj_torch[:, 0, :]**2 + ktraj_torch[:, 1, :]**2).unsqueeze(1)
dcomp = (rho / rho.max())
hanning = 0.5 + 0.5 * torch.cos(rho / rho.max())
dcomp_smooth = dcomp * hanning
kspace = signal.view(1, 1, -1)
kspace_real = kspace * dcomp_smooth
reco = adjnufft_ob(kspace_real, ktraj_torch)
reco_fixed = torch.rot90(reco.squeeze(), k=3, dims=(0, 1))
vmax = np.percentile(torch.abs(reco_fixed).numpy(), 99.99)
plt.imshow(torch.abs(reco_fixed), cmap='gray', vmin=0, vmax=vmax)
plt.title("Reconstructed GRE Image")
plt.show()